# Linear Regression

---
This notebook shows how to train, test and check a linear regression model in Python. It moves from a synthetic straight line, to a model with one real regressor, to a model with many regressors on bank marketing data, and finally to a forecasting model on Bitcoin returns.

**Learning objectives.** By the end of this notebook you should be able to:

1. Fit a linear regression with one regressor and read its two numbers (intercept and slope) as a plain business statement.
2. Build a model with many regressors from real data using only information that is known at decision time, and explain why anything observed later must be left out.
3. Judge a model on a test set by RMSE and R², compare both with a "predict the mean" baseline, and say what a low R² does and does not mean.
4. Check the five regression assumptions with the right plot or statistic for each, including a variance inflation factor (VIF) check for multicollinearity.
5. Explain why a linear model on lagged Bitcoin returns has almost no forecasting skill, and why regressing a price on yesterday's price gives an R² near 1 that means nothing.

**Data used.** `Admission_Predict.csv` (one regressor), `banking.csv` (many regressors) and `data_BTC.csv` (a daily time series). All three are read directly from the course repository on GitHub, so nothing has to be downloaded by hand.

**Note on importing libraries:**

General syntax to import specific functions from a library:
*from (library) import (specific function)*, e.g. *from pandas import DataFrame*

General syntax to import a whole library under a short alias:
*import (library) as (alias)*, e.g. *import matplotlib.pyplot as plt*, *import pandas as pd*

**Libraries:**

**Pandas** -- data manipulation and analysis (data frames, reading and writing files, slicing, merging, time series functionality).

**NumPy** -- arrays and matrices with a large collection of mathematical functions that operate on them.

**Matplotlib** -- the standard plotting library for Python.

**Seaborn** -- a data visualisation library built on top of matplotlib that makes statistical graphics easier to produce.

**scikit-learn** -- the standard machine learning library; we use it for train/test splitting, fitting the model and computing error metrics.

**statsmodels** -- a statistics library; we use it for the regression table with standard errors and p-values, for VIFs and for the Durbin-Watson statistic.


In [2]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
%matplotlib inline

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn import metrics


In [3]:
# To make this notebook's output stable across runs (we make the output reproducible)
np.random.seed(42)


In [4]:
# How to get help on a function: put a question mark after its name
print?


## A synthetic straight line

We start with data that we generate ourselves, so we know the true relationship: y = 4 + 3 x + noise. This lets us see what a linear model is supposed to recover before we move to real data, where the truth is unknown.


In [ ]:
# Let's generate some linear looking data:
# Note: numpy.random.randn generates samples from the normal distribution, while numpy.random.rand from the uniform
X = 2 * np.random.rand(100, 1)


In [ ]:
y = 4 + 3 * X + np.random.randn(100, 1) # notice a difference between the function to generate X and y? The former draws from a uniform distribution and the latter from a normal distribution.


In [ ]:
print(np.c_[X, y])  # Translates slice objects to concatenation along the second axis.


In [ ]:
# Let's plot (info on the marker and the color --> https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.plot.html)
plt.plot(X, y, "b.")
plt.xlabel("$x$", fontsize=18)
plt.ylabel("$y$", fontsize=18)
plt.axis([0, 2, 0, 12])
plt.show()


In [ ]:
# Training a linear model
lin_reg = LinearRegression() # create an object for the linear regression
lin_reg.fit(X, y) # fit the data
Y_predict = lin_reg.predict(X)
print('Intercept:', lin_reg.intercept_[0].round(2), ' Slope:', lin_reg.coef_[0][0].round(2), ' (true values: 4 and 3)')


In [ ]:
plt.plot(X, y, "b.")
plt.plot(X, Y_predict, color='red')
plt.show()


In [ ]:
X_new = np.array([[0.5], [1.75]])
y_predict = lin_reg.predict(X_new)
y_predict


## One real regressor: GRE score and chance of admission

---

`Admission_Predict.csv` has one row per applicant to a graduate programme and two columns:

- `GRE_score`: the applicant's score on the GRE test (290 to 340 in this file)
- `Admit`: the estimated chance of admission, a number between 0 and 1

With a single regressor the fitted model is a line, y = intercept + slope * x, and the slope has a direct reading: "one more GRE point is associated with `slope` more chance of admission". This is the last time in the notebook that the whole model fits into one sentence.


In [ ]:
admission_url = "https://raw.githubusercontent.com/umatter/EDFB/main/data/Admission_Predict.csv"
admission = pd.read_csv(admission_url)
print(admission.shape)
admission.describe()


In [ ]:
# Scatter plot: is a straight line a reasonable description?
plt.scatter(admission['GRE_score'], admission['Admit'], marker='.')
plt.xlabel('GRE score')
plt.ylabel('Chance of admission')
plt.show()


In [ ]:
# Fit the univariate model: Admit = intercept + slope * GRE_score
X_adm = admission[['GRE_score']]   # a DataFrame with one column (sklearn expects 2 dimensions)
y_adm = admission['Admit']
model_adm = LinearRegression()
model_adm.fit(X_adm, y_adm)

print('Intercept:', model_adm.intercept_)
print('Slope:', model_adm.coef_[0])
print('\nThe fitted model is Admit =', round(model_adm.intercept_, 3), '+', round(model_adm.coef_[0], 4), '* GRE_score')
print('R-squared (in sample):', round(model_adm.score(X_adm, y_adm), 3))


In [ ]:
# Plot data and fitted line
gre_grid = pd.DataFrame({'GRE_score': np.linspace(290, 340, 50)})
plt.scatter(admission['GRE_score'], admission['Admit'], marker='.', color='gray')
plt.plot(gre_grid['GRE_score'], model_adm.predict(gre_grid), color='red', linewidth=2)
plt.xlabel('GRE score')
plt.ylabel('Chance of admission')
plt.show()


**Reading the result.** In this run the slope is about 0.010: ten more GRE points go with roughly 0.10 (ten percentage points) more chance of admission. The intercept (about -2.4) is the predicted chance at a GRE score of 0, a value that does not occur, so it has no meaning on its own; it only positions the line. R² is about 0.64: GRE score alone accounts for roughly two thirds of the variation in admission chances across applicants.

**Assumption 2 (multicollinearity) for this model:** with a single regressor there is nothing for it to be collinear with, so the check is empty here. It becomes important in the next model, which has more than twenty regressors.


## Many regressors: predicting call duration in a bank's marketing campaign

---

**The data.** `banking.csv` is the bank marketing dataset from a Portuguese bank. One row is **one phone call** made during a marketing campaign for a term deposit (41,188 calls). The columns:

| column | meaning |
|---|---|
| `age`, `marital`, `education`, `housing`, `loan` | client characteristics (housing/personal loan: yes / no / unknown) |
| `contact` | how the client was reached: cellular or telephone (landline) |
| `previous` | number of contacts with this client before the current campaign |
| `pdays` | days since the client was last contacted in a *previous* campaign; **999 means never contacted before** (96% of rows) |
| `poutcome` | outcome of the previous campaign: success, failure, or nonexistent |
| `emp_var_rate`, `cons_price_idx`, `cons_conf_idx`, `euribor3m`, `nr_employed` | macroeconomic indicators at the time of the call: employment variation rate, consumer price index, consumer confidence index, 3-month Euribor rate, number of employees in the economy (thousands) |
| `campaign` | number of contacts during this campaign (including this call) |
| `duration` | length of the call **in seconds** |
| `y` | did the client subscribe to the term deposit? |

**The business question.** A call centre plans its staffing from the expected length of calls. Can we predict how long a call will last from what is known *before* the agent dials?

**The decision-time leakage rule.** Ask: *at the moment the decision is made, which columns are already known?* Before the call, the client's characteristics, the contact history and the macro indicators are known. `duration` is the target, `y` is only known at the end of the call, and `campaign` counts the current call itself, so these three are excluded from the regressors. Anything we would only learn during or after the call may not be used to predict it.

**Why `log1p(duration)`.** Call durations are heavily right-skewed (median 180 seconds, maximum 4,918 seconds). Modelling log(1 + duration) keeps the few very long calls from dominating the fit, and turns coefficients into approximate percentage effects: a coefficient of 0.10 means roughly 10% longer calls. `log1p` rather than `log` because a few calls have duration 0.

**`pdays = 999`.** The value 999 is a code for "never contacted", not a number of days. We split it into a flag `was_previously_contacted` and a cleaned `pdays_clean` in which the code is replaced by the median of the real values. That median is computed on the full data for simplicity; in a deployed pipeline it would be computed on the training rows only, so that nothing from the test rows enters the features.


In [ ]:
banking_url = "https://raw.githubusercontent.com/umatter/EDFB/main/data/banking.csv"
print("Fetching banking.csv from GitHub...")


In [ ]:
dataset = pd.read_csv(banking_url)


In [ ]:
dataset.head()


In [ ]:
dataset.tail()


In [ ]:
dataset.shape # Returns the dimensions of the array.


In [ ]:
dataset.dtypes # Returns the dtypes in the DataFrame.


In [ ]:
# Check for NAs
dataset.isna().any() # Generate a boolean mask indicating missing values


In [ ]:
dataset.isna().sum()


In [ ]:
# Describe the data
dataset.describe()


In [ ]:
print(dataset.columns)


In [ ]:
# Select the target and the regressors: predict log1p(duration) from pre-call client, contact-history and macro features.
# Leakage rule: 'duration' is the target, 'y' and 'campaign' are only known during or after the call, so they are excluded.
df_banking = dataset.copy()
df_banking['was_previously_contacted'] = (df_banking['pdays'] != 999).astype(int)
df_banking['pdays_clean'] = df_banking['pdays'].replace(999, np.nan)
df_banking['pdays_clean'] = df_banking['pdays_clean'].fillna(df_banking['pdays_clean'].median())   # median of the full data (simplification, see text)
df_banking['log_duration'] = np.log1p(df_banking['duration'])

feature_cols_cat = ['marital', 'education', 'housing', 'loan', 'contact', 'poutcome']
feature_cols_num = ['age', 'previous', 'pdays_clean', 'emp_var_rate', 'cons_price_idx', 'cons_conf_idx', 'euribor3m', 'nr_employed', 'was_previously_contacted']

# One dummy column per category level, dropping the first level of each variable (the reference category)
X_df = pd.get_dummies(df_banking[feature_cols_cat + feature_cols_num], drop_first=True).astype(float)

# 'housing' and 'loan' are 'unknown' for exactly the same 990 clients, so the two 'unknown' dummies are
# identical columns. Two identical regressors cannot both get a coefficient (perfect multicollinearity), so keep one.
X_df = X_df.drop(columns=['loan_unknown'])

X = X_df.values
y = df_banking['log_duration'].values
print('Regressors:', X_df.shape[1])
print(list(X_df.columns))

# Choose one feature for visualisation, by name
viz_col = 'age'

# Preserve banking variables so they are not overwritten by the Bitcoin section later on
X_df_banking = X_df
y_banking = y
viz_col_banking = viz_col


In [ ]:
# Scatter plot of the target against one feature (age)
plt.scatter(X_df_banking[viz_col_banking], y_banking, marker='.', alpha=0.2)
plt.xlabel(viz_col_banking)
plt.ylabel('log1p(duration)')
plt.show()


In [ ]:
# Correlations between the numeric variables (raw dataset, categorical columns removed)
corrmat = dataset.drop(['marital', 'education', 'housing', 'loan', 'contact', 'poutcome', 'y'], axis=1).corr()
corrmat.round(2)


In [ ]:
# Plot correlation heatmap
sns.heatmap(corrmat, cmap="YlGnBu", linewidths=0.1, annot=True, fmt='.2f')
plt.show()


The heatmap already shows one thing to remember for the assumption checks below: the macro indicators `emp_var_rate`, `euribor3m` and `nr_employed` are correlated at 0.9 or more with each other. They all measure the state of the economy at the time of the call, so they move together.


In [ ]:
# Split train and test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0) # 20% in testing; we set random_state, as every time you run it without specifying random_state, you will get a different result
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)
# Preserve banking split to avoid being overwritten by later BTC section
X_train_banking = X_train
X_test_banking = X_test
y_train_banking = y_train
y_test_banking = y_test


In [ ]:
# Fit the model on training set
model = LinearRegression()
model.fit(X_train, y_train) # training the algorithm


In [ ]:
# Get coefficients: one per regressor, so print them as a labelled table
print('Intercept:', round(model.intercept_, 3))
coef_table = pd.Series(model.coef_, index=X_df_banking.columns, name='coefficient')
print(coef_table.round(4).to_string())


There is no single "slope" any more: each coefficient is the change in log1p(duration) for a one-unit change in *that* regressor, holding the others fixed. For a dummy column the unit is "belongs to this category rather than the reference category". For example, in this run `contact_telephone` is roughly -0.2: calls to a landline are roughly 20% shorter than calls to a mobile, other things equal. Whether a coefficient is also *precisely estimated* is the subject of Exercise 1.


In [ ]:
# Get fitted value on test set
y_test_predicted = model.predict(X_test)
y_test_predicted_banking = y_test_predicted

# Compare predictions
pd.DataFrame({'True': y_test, 'Predicted': y_test_predicted}).head(10)


In [ ]:
# Plot predicted against true values on the test set. A perfect model would put every point on the dashed 45-degree line.
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_test_predicted, marker='.', alpha=0.2, color='gray')
lims = [y_test.min(), y_test.max()]
plt.plot(lims, lims, 'k--', label='45-degree line (perfect prediction)')
plt.xlabel('True log1p(duration)')
plt.ylabel('Predicted log1p(duration)')
plt.legend()
plt.show()


In [ ]:
# Plot some predicted vs true values against the visualisation feature (first 30 test observations)
points_to_plot = 30
viz_idx_banking = list(X_df_banking.columns).index(viz_col_banking)
x_subset = X_test[:points_to_plot, viz_idx_banking]
plt.scatter(x_subset, y_test[:points_to_plot], color='blue', marker='o', facecolors='none', label='true value')
plt.scatter(x_subset, y_test_predicted[:points_to_plot], color='red', marker='x', label='predicted')
plt.xlabel(viz_col_banking)
plt.ylabel('log1p(duration)')
plt.legend()
plt.show()


In [ ]:
# Evaluate Root Mean Square Error (RMSE), next to the scale of the target and next to a mean-only baseline
RMSE_test = np.sqrt(metrics.mean_squared_error(y_test, y_test_predicted))
RMSE_mean_only = np.sqrt(np.mean((y_test - y_train.mean())**2))   # always predict the training mean
print('Root Mean Squared Error on test set:', round(RMSE_test, 4))
print('RMSE of predicting the training mean:', round(RMSE_mean_only, 4))
print('Mean of log1p(duration) in y_test:', round(y_test.mean(), 4))
print('Standard deviation of log1p(duration) in y_test:', round(y_test.std(), 4))


In [ ]:
# Evaluate R-squared on the test set: 1 - SSE / SST
R2 = metrics.r2_score(y_test, y_test_predicted)
print('R-squared on test set:', round(R2, 4))
print('R-squared on training set:', round(model.score(X_train, y_train), 4))


**What these numbers say.** In this run the test R² is about 0.013: the model explains roughly 1% of the variation in log call duration. The RMSE (about 0.91) is within one percent of the standard deviation of the target, and only a hair below the RMSE of a model that always predicts the training mean. On the log scale an error of 0.9 means the typical prediction is off by a factor of about 2.5 in either direction.

**Why so low, and why that is normal.** How long a call lasts depends mostly on what happens *during* the call: whether the client is interested, asks questions, or hangs up. None of that is known before dialling, and the leakage rule says we may not use it. Pre-call characteristics carry only a little information about call length. A low R² here is not a bug in the code; it is the honest answer to the question "how predictable is call length from what we know in advance?", and that answer ("hardly at all") is itself useful for the call centre: it should plan on the average with a wide margin rather than trust per-call predictions.

**Two things a low R² does not mean.** It does not mean the coefficients are wrong or unimportant: with 33,000 training calls, several of them are estimated precisely (Exercise 1), and they tell us which client and contact features shift call length on average. And it does not mean a different model would do much better; Exercise 3 and Exercise 7 will show that adding features or regularising changes little, because the missing information is not in the data.

Note that `np.corrcoef(y_test, y_test_predicted)[0, 1] ** 2`, which is sometimes used as "R²", is a different quantity: it is always at least zero and ignores whether the predictions are biased or on the wrong scale. The 1 - SSE/SST definition used here (`r2_score`) can be negative on a test set (see the Bitcoin section).


## Check linear regression assumptions

**Linear regression assumes the following:**

---

1. a **linear relationship** between each regressor and the target
2. **no perfect multicollinearity** between regressors, and not too much strong multicollinearity
3. **homoscedasticity**: the variance of the error terms (residuals) does not depend on the fitted value or on the regressors
4. **normally distributed error terms** (this matters for confidence intervals and p-values in small samples, not for the predictions themselves)
5. **independent error terms**: for time series data this means **no autocorrelation** in the residuals, i.e. no correlation between $e_t$ and $e_{t-1}$

You will sometimes see "no correlation between regressors and residuals" (exogeneity) listed as an assumption. It cannot be checked from the residuals of the fitted model: on the training data, ordinary least squares makes those correlations exactly zero by construction, so the check would always pass. Exogeneity is a statement about how the data came about (no omitted variables that move with the regressors) and has to be argued, not plotted.

Assumptions 1 to 4 are checked on the banking model here. Assumption 5 is a statement about the *order* of observations; the banking rows are individual calls in arbitrary order, so there is nothing to check. We check it on the Bitcoin model below, where the order is time.


In [ ]:
# Checking Assumption 1 - linear relationship between a regressor and the target
# With 41,000 calls a raw scatter is a solid block, so plot the mean of log1p(duration) at each age.
mean_by_age = df_banking.groupby('age')['log_duration'].agg(['mean', 'count'])
plt.scatter(X_df_banking[viz_col_banking], y_banking, marker='.', alpha=0.05, color='gray', label='individual calls')
plt.plot(mean_by_age.index, mean_by_age['mean'], color='red', linewidth=2, label='mean log1p(duration) at each age')
plt.title('Feature vs log1p(duration)')
plt.xlabel(viz_col_banking)
plt.ylabel('log1p(duration)')
plt.legend()
plt.show()


The mean of the target moves very little with age and roughly along a line; the few ages above 80 have only a handful of calls each, which is why the red line jumps around there. Nothing here calls for a curved term in age.

**Checking Assumption 2 - little to no multicollinearity between regressors**

Multicollinearity means that one regressor can be predicted well from the others. It does not hurt predictions, but it makes the individual coefficients unstable and their standard errors large: the model cannot tell which of two near-identical regressors deserves the credit. The variance inflation factor (VIF) of a regressor is 1 / (1 - R²) of the regression of that regressor on all the others. VIF = 1 means no collinearity; values above 5 are worth noting and above 10 are usually called high.


In [ ]:
# Checking Assumption 2 - VIF for every regressor of the banking model
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_vif = sm.add_constant(X_train_banking)   # VIF is computed with an intercept in the auxiliary regressions
vif = pd.Series([variance_inflation_factor(X_vif, i) for i in range(1, X_vif.shape[1])],
                index=X_df_banking.columns, name='VIF').sort_values(ascending=False)
print(vif.round(1).to_string())


In this run the three macro indicators have VIFs between about 30 and 65 and `emp_var_rate`, `euribor3m` and `nr_employed` are the culprits: as the heatmap showed, they measure the same thing (the state of the economy) and move together. `was_previously_contacted` and `poutcome_success` are also collinear (VIF around 13 to 14), because a previous campaign can only have succeeded for clients who were contacted before. Everything else is below 10.

What follows from that: the predictions are fine, but the *individual* coefficients of the three macro variables should not be read separately. A safe reading is "a better economy (higher rates, more employment) goes with shorter calls"; splitting that into "Euribor does X and employment does Y" is not supported. If we had left both `unknown` dummies in the model, one VIF would have been infinite (perfect multicollinearity); that is why the construction cell dropped `loan_unknown`.


In [ ]:
# Checking Assumption 3 - Homoscedasticity
# Plot the residuals against the fitted values. Under homoscedasticity the vertical spread of the points is the same
# for small and large fitted values. A funnel (narrow on one side, wide on the other) indicates heteroscedasticity.
residuals_test = y_test_banking - y_test_predicted_banking
plt.figure(figsize=(10, 6))
plt.scatter(y_test_predicted_banking, residuals_test, marker='o', facecolors='none', color='black', alpha=0.3)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Fitted values (predicted log1p(duration))')
plt.ylabel('Residuals (true - predicted)')
plt.title('Residuals vs fitted values')
plt.show()


The spread of the residuals is about the same across the range of fitted values, so there is no funnel. Note the diagonal edge at the bottom left: it comes from the few calls with duration 0, whose residual is 0 minus the fitted value.


In [ ]:
# Checking Assumption 4 - Normal distribution of residuals
# Check if the residual distribution looks like a normal distribution with the same mean and variance

resid_mean = residuals_test.mean()
resid_std = residuals_test.std()
normal_distr = np.random.normal(resid_mean, resid_std, len(residuals_test))

fig, ax = plt.subplots(1, 3, sharex='col', sharey='row', figsize=(20, 6))
sns.histplot(residuals_test, kde=True, stat='density', ax=ax[0])
ax[0].set_title('Residual distribution', fontsize=20)
sns.histplot(normal_distr, kde=True, stat='density', ax=ax[1], color='orange')
ax[1].set_title('Normal distribution', fontsize=20)
sns.histplot(residuals_test, kde=True, stat='density', label='residuals', ax=ax[2])
sns.histplot(normal_distr, kde=True, stat='density', label='normal\ndistribution', ax=ax[2], color='orange')
ax[2].legend(loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize=20)
plt.show()


In [ ]:
# Check Q-Q plot: the quantiles of the residuals against the theoretical quantiles of a normal distribution
# with the same mean and standard deviation. If the residuals were normal, the points would lie on the dashed line.
fig = sm.qqplot(residuals_test, line='45', fit=True, marker='o', markerfacecolor='none', markeredgecolor='blue')
plt.xlabel('theoretical normal quantiles', fontsize=16)
plt.ylabel('residual quantiles', fontsize=16)
plt.show()


The residuals are close to normal in the centre and have a heavier left tail (the very short calls). With 8,000 test observations this does not affect the predictions or the coefficient estimates; it would only matter for confidence intervals in a very small sample.

**Assumption 5 (independence, no autocorrelation)** is checked in the Bitcoin section, where the observations have a time order.


## Linear regression for forecasting: Bitcoin

---

In this section we use a linear model to **forecast** something: tomorrow's Bitcoin return from the returns of the last five days. The data is `data_BTC.csv`, one row per day with the closing price in US dollars (`BTC-USD.Close`) from August 2017 to September 2026.

Two ideas make this section different from the banking model:

1. **The order of the rows is time.** We may only use the past to predict the future, so the train/test split is chronological (first 80% of days for training, last 20% for testing), and every regressor for day *t* must be something that was known at the end of day *t - 1*.
2. **We forecast returns, not prices.** The first model below shows why: regressing today's price on yesterday's price gives an R² near 1 that means nothing.


In [ ]:
# Let's import the dataset including Bitcoin prices
btc_url = "https://raw.githubusercontent.com/umatter/EDFB/main/data/data_BTC.csv"
print("Fetching data_BTC.csv from GitHub...")


In [ ]:
data = pd.read_csv(btc_url)
data['Date'] = pd.to_datetime(data['Date'])
# Ensure strict chronological order
data = data.sort_values('Date').reset_index(drop=True)


In [ ]:
# Let's check if we imported correctly
data.head()


In [ ]:
# Let's get some summary stats on the prices
data.describe()


In [ ]:
# Let's plot the movement of the BTC Close Price
plt.figure(figsize=(12, 6))
plt.plot(data['Date'], data['BTC-USD.Close'], label='Bitcoin Price', color='blue')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.title('Bitcoin Price Over Time')
plt.legend()
plt.show()


### The trap: regressing price on lagged price

The obvious first idea is to predict today's price from yesterday's price. Let us do that, and also compare it with the simplest possible "forecast": just say that today's price equals yesterday's price.


In [ ]:
# Price on lagged price, chronological split
price_df = pd.DataFrame({'price': data['BTC-USD.Close'], 'price_lag_1': data['BTC-USD.Close'].shift(1)}).dropna()
split_p = int(len(price_df) * 0.8)
train_p, test_p = price_df.iloc[:split_p], price_df.iloc[split_p:]

model_price = LinearRegression().fit(train_p[['price_lag_1']], train_p['price'])
pred_price = model_price.predict(test_p[['price_lag_1']])

print('Fitted model: price_t =', round(model_price.intercept_, 2), '+', round(model_price.coef_[0], 4), '* price_{t-1}')
print('R-squared on test set (model):        ', round(r2_score(test_p['price'], pred_price), 4))
print('R-squared on test set ("price_t = price_{t-1}", no model at all):', round(r2_score(test_p['price'], test_p['price_lag_1']), 4))


In this run the model has a test R² of about 0.99, and its slope is about 1.00. But "tomorrow's price is today's price", which requires no model and no data, has the *same* R². The high R² comes entirely from the fact that prices wander slowly: tomorrow's price is always close to today's, whatever happens next. The model has learned nothing about *where* the price is going. A price series is **non-stationary** (its level drifts and its variance grows over time), and a regression on a non-stationary series produces exactly this kind of impressive-looking, useless fit.

### Forecasting returns instead

What a trader actually needs is the *change*: the **log return** $r_t = \log(P_t) - \log(P_{t-1})$, roughly the percentage change of the price. Returns are close to stationary (they hover around zero with a roughly stable spread), so a regression of $r_t$ on past returns $r_{t-1}, \dots, r_{t-5}$ is a fair test of whether the past contains information about the future. Set your expectation now: if markets are even roughly efficient, the answer will be "almost none", and the test R² will be around zero. That is the honest result to look for, not a failure of the method.


In [ ]:
# Create lagged return features. This function does not mutate its input DataFrame.
def create_lagged_features(data, lag):
    df = data.copy()
    df['ret'] = np.log(df['BTC-USD.Close']).diff()      # log return of day t
    for i in range(1, lag+1):
        df[f'lag_ret_{i}'] = df['ret'].shift(i)          # the return i days earlier: known at the end of day t-1
    df = df.dropna().reset_index(drop=True)
    return df


In [ ]:
# Create lag features on log returns with 5 lags
data = create_lagged_features(data, lag=5)


In [ ]:
data.head()


**Which columns may be regressors?** Only the five lagged returns. `ret` is the target. `BTC-USD.Close` is the closing price of day *t* itself, which is not known until the day is over; using it (or any open, high, low or volume figure of the same day) would let the model see the answer. The rule is the same as in the banking model: nothing observed during period *t* may be used to predict period *t*.


In [ ]:
# Split the data into training and testing sets using a chronological split (avoid leakage)
feature_cols_btc = [f'lag_ret_{i}' for i in range(1, 6)]
X_btc = data[feature_cols_btc]
y_btc = data['ret']
split_idx = int(len(data) * 0.8)
X_train_btc, X_test_btc = X_btc.iloc[:split_idx], X_btc.iloc[split_idx:]
y_train_btc, y_test_btc = y_btc.iloc[:split_idx], y_btc.iloc[split_idx:]
print('Training days:', len(X_train_btc), ' Test days:', len(X_test_btc), ' Test period starts:', data['Date'].iloc[split_idx].date())


In [ ]:
# Train a linear regression model with statsmodels (add an intercept), which reports standard errors and p-values
X_train_btc_sm = sm.add_constant(X_train_btc, has_constant='add')
X_test_btc_sm = sm.add_constant(X_test_btc, has_constant='add')
model_btc = sm.OLS(y_train_btc, X_train_btc_sm).fit()

# Print the summary, which includes coefficients and p-values
print(model_btc.summary())


**How to read this table.** The block in the middle has one row per regressor:

- `coef` is the estimated coefficient, as before.
- `std err` is the standard error: how much the coefficient would wobble if we re-estimated it on a different sample of days. A coefficient is *precisely estimated* when it is several times its standard error.
- `t` is coef / std err, and `P>|t|` is the p-value: the probability of seeing a coefficient at least this far from zero if the true coefficient were zero. A small p-value (say below 0.05) is evidence that the coefficient is not exactly zero. It is **not** a measure of importance: with thousands of observations even a tiny effect gets a small p-value.
- `[0.025  0.975]` is the 95% confidence interval for the coefficient.

Above the block, `R-squared` is the in-sample R², here about 0.006 in this run: the five lags explain about half a percent of the variation of daily returns. In this run `lag_ret_1` and `lag_ret_2` have p-values around 0.01 to 0.02, so there is weak statistical evidence of a small negative first-order and positive second-order dependence. Statistically detectable and economically useful are different things: a "significant" lag inside an R² of 0.006 is of no use for trading, as the test set will show. The `Durbin-Watson` line at the bottom is the autocorrelation check of Assumption 5, discussed below.


In [ ]:
# Make predictions on the test set
y_pred_btc = model_btc.predict(X_test_btc_sm)

# Root Mean Squared Error of the model
rmse = np.sqrt(mean_squared_error(y_test_btc, y_pred_btc))

# Benchmark: the zero forecast ("no change", tomorrow's expected return is 0). A model with forecasting skill must beat it.
rmse_zero = np.sqrt(np.mean(y_test_btc**2))

# A tempting but wrong benchmark: predict tomorrow's return with today's return (lag 1).
# Returns are close to noise, so this benchmark doubles the error variance and makes any model look 30% better than it is.
rmse_lag1 = np.sqrt(mean_squared_error(y_test_btc, X_test_btc['lag_ret_1']))

# R-squared on the test set: 1 - SSE / SST
r2 = r2_score(y_test_btc, y_pred_btc)

print(f"RMSE (model):                    {rmse:.6f}")
print(f"RMSE (zero forecast):            {rmse_zero:.6f}")
print(f"RMSE ratio (model / zero):       {rmse/rmse_zero:.3f}   (below 1 would mean forecasting skill)")
print(f"RMSE (lag-1 'benchmark'):        {rmse_lag1:.6f}")
print(f"RMSE ratio (model / lag-1):      {rmse/rmse_lag1:.3f}   (looks good, but only because the benchmark is bad)")
print(f"R-squared on test set:           {r2:.4f}")
print(f"Standard deviation of test returns: {y_test_btc.std():.6f}")


**Reading the result.** In this run the model's RMSE is 1.003 times the RMSE of the zero forecast and the test R² is about -0.006. A model that cannot beat "no change" has no forecasting skill, and this one cannot. Against the lag-1 "benchmark" the ratio is about 0.69, which looks like a 30% improvement; it is not, because predicting a noisy series by its own last value is a much worse forecast than predicting zero (its error variance is twice as large). Always compare a return forecast with the zero or the training-mean forecast.

**Why can test R² be negative?** R² on a test set is 1 minus (sum of squared errors of the model) divided by (sum of squared deviations from the test mean). Nothing forces the model to do better than the test mean on data it has not seen, and when it does worse, R² goes below zero. A test R² of -0.006 means the model is a little worse than a flat line at the test-set average. Squared correlation, which is sometimes used instead, is always at least zero and would hide this.


In [ ]:
# A few predictions next to the true values (daily returns are around 0.02 in size, so use 4 decimals)
pd.DataFrame({'Date': data['Date'].iloc[split_idx:].dt.date.values,
              'True': y_test_btc.values.round(4),
              'Predicted': y_pred_btc.values.round(4)}).head(10)


In [ ]:
# Plot true vs predicted returns over time. The predictions are a nearly flat line: the model has almost nothing to say.
plt.figure(figsize=(12, 5))
plt.plot(data['Date'].iloc[split_idx:], y_test_btc, label='True returns', alpha=0.7)
plt.plot(data['Date'].iloc[split_idx:], y_pred_btc, label='Predicted returns', alpha=0.9)
plt.legend()
plt.tight_layout()
plt.show()


### Checking Assumption 5 on the Bitcoin model: no autocorrelation in the residuals

Here the observations are ordered in time, so the assumption has content. If the residuals of day *t* were correlated with those of day *t - 1*, the model would be leaving predictable structure on the table. Two standard checks:

- the **Durbin-Watson statistic** $DW = \sum (e_t - e_{t-1})^2 / \sum e_t^2$, which is 2 when there is no first-order autocorrelation, below 2 for positive and above 2 for negative autocorrelation;
- the **autocorrelation function (ACF)** of the residuals: the correlation of $e_t$ with $e_{t-k}$ for k = 1, 2, ..., plotted with a band outside which a correlation would be more than noise.


In [ ]:
from statsmodels.stats.stattools import durbin_watson
from statsmodels.graphics.tsaplots import plot_acf

resid_btc = model_btc.resid
print('Durbin-Watson statistic (training residuals):', round(durbin_watson(resid_btc), 3), ' (2 = no first-order autocorrelation)')

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(y_train_btc, lags=20, ax=ax[0], title='ACF of the returns themselves')
plot_acf(resid_btc, lags=20, ax=ax[1], title='ACF of the model residuals')
plt.show()


In this run the Durbin-Watson statistic is 2.00 and the residual autocorrelations are all inside the noise band. The left panel shows why the model had so little to work with: the returns themselves are almost uncorrelated from one day to the next (the lag-1 and lag-2 correlations are around 0.05 in size), and the model has absorbed that small amount. What is left is white noise, which is exactly what weak-form market efficiency predicts.


## Exercises

Now it's time to practice. Exercises marked **core** are the ones to do in class; **optional** ones go deeper. Each exercise names the deliverable to hand in.

### Exercise 1 (core): Which coefficients are large and precisely estimated?

**Task:** Interpret the coefficients of the banking model. A coefficient is worth a business sentence only if it is both large enough to matter *and* estimated precisely enough to trust.

**Instructions:**
1. Refit the banking model with `statsmodels` (`sm.OLS(y_train_banking, sm.add_constant(X_train_banking))`) so that you get standard errors and confidence intervals; label the rows with `X_df_banking.columns`.
2. List the coefficients sorted by absolute size, together with their standard error and 95% confidence interval.
3. Pick the five coefficients that are both large (say, |coef| > 0.05, i.e. more than a 5% effect on call duration) and precisely estimated (confidence interval does not contain 0). Which large coefficients fail the second test?
4. Explain what the five mean in business terms (how they shift call duration), remembering the VIF result: the macro variables cannot be read one by one.

**Deliverable:** a table of the five coefficients with their confidence intervals and one sentence per coefficient on its business meaning.


In [ ]:
# Exercise 1: Your code here
# Hint: model_sm = sm.OLS(y_train_banking, sm.add_constant(X_train_banking)).fit()
#       model_sm.params, model_sm.bse and model_sm.conf_int() are arrays in the order of X_df_banking.columns

# Step 1: refit with statsmodels to get standard errors and confidence intervals (sklearn's LinearRegression doesn't report these)
X_train_sm = sm.add_constant(X_train_banking)   # prepends a column of 1s for the intercept, as the FIRST column
model_sm = sm.OLS(y_train_banking, X_train_sm).fit()

# Step 2: build a labelled coefficient table.
# X_train_banking is a plain numpy array (no column names), so params/bse/conf_int come back unlabelled too.
# We attach labels ourselves, in the same order: 'const' first (because add_constant prepends it), then X_df_banking.columns.
coef_labels = ['const'] + list(X_df_banking.columns)
ci = model_sm.conf_int()   # shape (n_params, 2): column 0 = lower bound, column 1 = upper bound of the 95% CI

coef_table = pd.DataFrame({
    'coef': model_sm.params,
    'std_err': model_sm.bse,
    'ci_lower': ci[:, 0],
    'ci_upper': ci[:, 1],
}, index=coef_labels)

# Step 3: sort by the SIZE of the effect (absolute value), largest first
coef_table_sorted = coef_table.reindex(coef_table['coef'].abs().sort_values(ascending=False).index)
print('All coefficients, sorted by |coef|:')
print(coef_table_sorted.round(4).to_string())

# Step 4: "large" = |coef| > 0.05 (roughly a 5% effect on call duration, since the target is log1p(duration))
#         "precisely estimated" = the 95% CI does not straddle 0, i.e. ci_lower and ci_upper have the same sign
is_large = coef_table_sorted['coef'].abs() > 0.05
is_precise = (coef_table_sorted['ci_lower'] * coef_table_sorted['ci_upper']) > 0   # same sign => product is positive

large_and_precise = coef_table_sorted[is_large & is_precise]
large_but_imprecise = coef_table_sorted[is_large & ~is_precise]

print('\nLarge AND precisely estimated (top 5):')
print(large_and_precise.head(5).round(4).to_string())

print('\nLarge but NOT precisely estimated (CI contains 0) -- these fail the second test:')
print(large_but_imprecise.round(4).to_string() if len(large_but_imprecise) else '(none)')

# Reminder from the VIF check earlier: these three move together (VIF 30-65), so if any of them show up
# in the top 5, they must be interpreted as a bundle ("a better economy"), not individually.
macro_vars = {'emp_var_rate', 'euribor3m', 'nr_employed'}
flagged = set(large_and_precise.head(5).index) & macro_vars
if flagged:
    print(f'\nNote: {flagged} are in the top 5 but collinear with each other -- read them as one "macro" effect, not separately.')


**Deliverable: the five large and precisely estimated coefficients**

| Variable | coef | 95% CI | ≈ % effect on duration |
|---|---|---|---|
| `contact_telephone` | -0.2147 | (-0.244, -0.185) | ≈ 19% shorter |
| `euribor3m` | 0.1803 | (0.135, 0.226) | ≈ 20% longer per 1pt |
| `cons_price_idx` | 0.1461 | (0.095, 0.198) | ≈ 16% longer per 1pt |
| `emp_var_rate` | -0.1262 | (-0.165, -0.088) | ≈ 12% shorter per 1pt |
| `education_university.degree` | -0.0775 | (-0.115, -0.041) | ≈ 7% shorter |

(% effect computed as exp(coef) - 1, not the raw coefficient, since at |coef| ≈ 0.2 the two start to diverge by a couple of points.)

1. **`contact_telephone`**: calls reached on a landline run about 19% shorter than calls on mobile, holding client and macro conditions fixed — worth checking whether landline reach itself correlates with less-engaged (often older, listed) contacts.
2. **`euribor3m`**: a one-point-higher 3-month Euribor rate goes with roughly 20% longer calls — but this cannot be attributed to Euribor alone; see the caveat below.
3. **`cons_price_idx`**: a one-point-higher consumer price index goes with roughly 16% longer calls. Unlike the other two macro variables, this one has low VIF (not part of the collinear trio), so it can be read on its own.
4. **`emp_var_rate`**: a one-point-higher employment variation rate goes with roughly 12% *shorter* calls — again cannot be read alone; see below.
5. **`education_university.degree`**: clients with a university degree have calls about 7% shorter than the reference group (`basic.4y` education), other things equal — possibly reflecting quicker decision-making or lower need for explanation.

**Caveat (multicollinearity).** `euribor3m` and `emp_var_rate` are two of the three macro variables the earlier VIF check flagged (VIF 30-65, along with `nr_employed`); they all measure "the state of the economy" and move together, so #2 and #4 cannot be reported as two independent effects. The honest single sentence is: *a stronger macro environment (higher rates, higher employment) is associated with shorter calls overall*, not "Euribor does X and employment does Y separately." `cons_price_idx` (#3) is a different macro indicator with its own low VIF, so it genuinely stands alone.

**Large coefficients that failed the precision test** (their 95% CI crosses zero, so we cannot rule out no real effect): `was_previously_contacted` (0.171, CI -0.03 to 0.37), `marital_unknown` (0.134, CI -0.09 to 0.36), `poutcome_success` (0.072, CI -0.12 to 0.27), and `education_illiterate` (0.062, CI -0.39 to 0.51). These all correspond to small or rare subgroups of clients, which inflates the standard error even when the point estimate looks sizeable — a textbook "large but noisy" result, not something to act on.

The intercept (`const` ≈ 4.79, CI -3.0 to 12.6) is both imprecise and meaningless on its own: it is the predicted value at age 0 and every dummy at its reference level, which describes no real client — the same caveat the notebook raised for the intercept of the GRE admission model earlier.

### Exercise 2 (core): Residual Analysis

**Task:** Perform a residual analysis of the banking model to check the assumptions.

**Instructions:**
1. Calculate residuals for both the training and the test set
2. Create a residuals vs fitted values plot
3. Check the normality of the residuals with a histogram and a Q-Q plot
4. Look for patterns that would indicate an assumption violation

**Deliverable:** a 2 x 2 panel of diagnostic plots and one sentence per panel saying what it shows.


In [ ]:
# Exercise 2: Your code here
# Hint: Calculate residuals = actual - predicted
# Use subplots to create multiple diagnostic plots

# Step 1: residuals for BOTH the training and the test set, using the already-fitted model from earlier (`model`)
y_train_pred_ex2 = model.predict(X_train_banking)
y_test_pred_ex2 = model.predict(X_test_banking)
residuals_train_ex2 = y_train_banking - y_train_pred_ex2
residuals_test_ex2 = y_test_banking - y_test_pred_ex2

# Step 2: a 2x2 panel of diagnostic plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel (0,0): residuals vs fitted -- TRAINING set (checks homoscedasticity, Assumption 3)
axes[0, 0].scatter(y_train_pred_ex2, residuals_train_ex2, marker='o', facecolors='none', color='black', alpha=0.1)
axes[0, 0].axhline(0, color='red', linestyle='--')
axes[0, 0].set_xlabel('Fitted values (train)')
axes[0, 0].set_ylabel('Residuals (train)')
axes[0, 0].set_title('Residuals vs fitted -- training set')

# Panel (0,1): residuals vs fitted -- TEST set (same check, out-of-sample)
axes[0, 1].scatter(y_test_pred_ex2, residuals_test_ex2, marker='o', facecolors='none', color='black', alpha=0.3)
axes[0, 1].axhline(0, color='red', linestyle='--')
axes[0, 1].set_xlabel('Fitted values (test)')
axes[0, 1].set_ylabel('Residuals (test)')
axes[0, 1].set_title('Residuals vs fitted -- test set')

# Panel (1,0): histogram of TEST residuals against a normal curve with the same mean/std (checks Assumption 4)
resid_mean_ex2 = residuals_test_ex2.mean()
resid_std_ex2 = residuals_test_ex2.std()
normal_ref_ex2 = np.random.normal(resid_mean_ex2, resid_std_ex2, len(residuals_test_ex2))
sns.histplot(residuals_test_ex2, kde=True, stat='density', ax=axes[1, 0], label='residuals (test)')
sns.histplot(normal_ref_ex2, kde=True, stat='density', ax=axes[1, 0], color='orange', alpha=0.5, label='normal reference')
axes[1, 0].set_title('Residual distribution vs normal -- test set')
axes[1, 0].legend()

# Panel (1,1): Q-Q plot of TEST residuals (same check, more sensitive to tail behaviour than the histogram)
sm.qqplot(residuals_test_ex2, line='45', fit=True, ax=axes[1, 1], marker='o', markerfacecolor='none', markeredgecolor='blue')
axes[1, 1].set_title('Q-Q plot -- test set residuals')

plt.tight_layout()
plt.show()

# Numeric support for the written commentary
print('Train residuals: mean =', round(residuals_train_ex2.mean(), 6), ' std =', round(residuals_train_ex2.std(), 4))
print('Test residuals:  mean =', round(residuals_test_ex2.mean(), 6), ' std =', round(residuals_test_ex2.std(), 4))


**Deliverable: 2x2 diagnostic panel, one sentence per panel**

This repeats the checks the notebook already ran for Assumptions 3 and 4 (cells checking homoscedasticity and normality), but adds the training-set view and puts all four in one panel. Since it's the identical model and data, the interpretation below follows directly from what those earlier cells already established — confirm by eye once you've run it, since the exact numbers will differ slightly train vs. test:

1. **Residuals vs fitted — training set (top left).** By construction, OLS residuals on the *training* data average to zero and are uncorrelated with the fitted values — that's not a pattern to look for, it's a guaranteed property of how least squares is fit. What to actually check here is the *spread*: it should look like a horizontal band of roughly constant width, not a funnel. Given the training set is ~4x larger than the test set (33k vs. 8k rows), expect a denser but similarly shaped cloud to the test panel.
2. **Residuals vs fitted — test set (top right).** This is the genuine out-of-sample check, since these residuals were *not* used to fit the line. The earlier assumption-check section already showed this exact plot for this exact model: no funnel shape, roughly constant spread across the range of fitted values — homoscedasticity looks fine. The one visible feature is a diagonal edge at the bottom-left, coming from the handful of calls with `duration = 0` (their residual is simply `0 − fitted`).
3. **Histogram vs normal — test set (bottom left).** The earlier normality check found the residuals close to normal in the centre with a heavier *left* tail (from the cluster of very short calls near duration 0, which the log1p transform can't fully symmetrize). Expect the same shape here.
4. **Q-Q plot — test set (bottom right).** Consistent with the histogram: points should sit close to the 45-degree line through the middle of the distribution, curving away from it at the lower-left tail (the short-call cluster again). The Q-Q plot is the more sensitive of the two normality checks for exactly this kind of tail deviation.

**Overall (instruction 4 — do the patterns indicate an assumption violation?)** No violation serious enough to worry about at this sample size. The mild left-tail non-normality doesn't bias the coefficients or predictions — the notebook's own text on this point: with thousands of observations, deviations from normality mainly affect confidence intervals in *small* samples, which doesn't apply here. If you see a funnel shape or a sharply different pattern between the train and test panels when you actually run this, that would be the one thing to flag back to me — it would mean something changed between the two samples that this analysis didn't anticipate.

### Exercise 3 (optional): Feature Engineering

**Task:** Create new features and see whether they improve the banking model.

**Business Context:** Sometimes combining existing features or creating polynomial terms can capture non-linear relationships.

**Instructions:**
1. Create interaction terms between two numerical features
2. Add polynomial features (squared terms) for numerical variables, and keep the dummy columns in the model so that the comparison is fair
3. Train a new model with these engineered features
4. Compare R² and RMSE on the test set with the original model

**Deliverable:** a table of test R² and RMSE for the original and the engineered model, and one sentence on whether the extra features helped.


In [ ]:
# Exercise 3: Your code here
# Hint: Use sklearn.preprocessing.PolynomialFeatures for polynomial terms
# Be careful about overfitting with too many features

from sklearn.preprocessing import PolynomialFeatures

# Step 1: pick two numerical features to engineer on. age and pdays_clean are a natural pair:
# does the effect of age on call length depend on how recently the client was last contacted?
num_interact_cols = ['age', 'pdays_clean']

# degree=2 with include_bias=False gives: age, pdays_clean, age^2, age*pdays_clean, pdays_clean^2
poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=False)
poly_features = poly.fit_transform(X_df_banking[num_interact_cols])
poly_feature_names = poly.get_feature_names_out(num_interact_cols)

# Keep only the NEW engineered columns (drop age, pdays_clean themselves -- they're already in X_df_banking)
new_cols_mask = [name not in num_interact_cols for name in poly_feature_names]
new_col_names = [n for n, keep in zip(poly_feature_names, new_cols_mask) if keep]
X_engineered_new = pd.DataFrame(poly_features[:, new_cols_mask], columns=new_col_names, index=X_df_banking.index)

# Step 2: combine with the ORIGINAL feature set (all dummies + all original numeric columns stay in, per instructions)
X_df_engineered = pd.concat([X_df_banking, X_engineered_new], axis=1)
X_eng = X_df_engineered.values
y_eng = y_banking

# Step 3: SAME random_state and test_size as the original split, so this is an apples-to-apples comparison
X_train_eng, X_test_eng, y_train_eng, y_test_eng = train_test_split(X_eng, y_eng, test_size=0.2, random_state=0)

model_eng = LinearRegression()
model_eng.fit(X_train_eng, y_train_eng)
y_test_pred_eng = model_eng.predict(X_test_eng)

# Step 4: compare test R2 and RMSE against the ORIGINAL model (already fitted earlier as `model`)
r2_original = r2_score(y_test_banking, y_test_predicted_banking)
rmse_original = np.sqrt(mean_squared_error(y_test_banking, y_test_predicted_banking))
r2_engineered = r2_score(y_test_eng, y_test_pred_eng)
rmse_engineered = np.sqrt(mean_squared_error(y_test_eng, y_test_pred_eng))

comparison = pd.DataFrame({
    'model': ['original', 'engineered (+ age^2, pdays_clean^2, age*pdays_clean)'],
    'n_features': [X_df_banking.shape[1], X_df_engineered.shape[1]],
    'test_R2': [r2_original, r2_engineered],
    'test_RMSE': [rmse_original, rmse_engineered],
})
print(comparison.round(4).to_string(index=False))
print('\nNew engineered columns added:', new_col_names)


**Deliverable: does feature engineering help?**

The code adds three engineered columns on top of the original 22 features: `age^2`, `pdays_clean^2`, and `age*pdays_clean` (the interaction), refit on the identical train/test split so the comparison is fair.

**Expected outcome, and why.** The notebook's own "Key Takeaways" section already told us what to expect here, back when it discussed the original model's low R²: *"it does not mean a different model would do much better; Exercise 3 and Exercise 7 will show that adding features or regularising changes little, because the missing information is not in the data."* The reasoning: call duration is overwhelmingly decided by what happens *during* the call (client interest, questions, hang-ups) — none of that is in any pre-call feature, no matter how it's transformed. Squaring `age` or interacting it with `pdays_clean` cannot manufacture information about in-call behaviour that was never captured. So the honest prediction is: test R² moves by at most a few thousandths, and RMSE barely changes — not because the code is wrong, but because the ceiling on predictability was already reached by the original model.

**One sentence (template — finalize with your real numbers):** *"Adding [interaction/polynomial] features changed test R² from [original] to [engineered] and RMSE from [original] to [engineered], confirming that additional feature complexity does not help because the missing predictive information (in-call behaviour) simply is not present in any pre-call variable, however it is transformed."*

Run the cell above and send me the printed comparison table — I'll fill in the real numbers and confirm (or correct) this prediction rather than leave a guessed table in the deliverable.

### Exercise 4 (core): Cross-Validation

**Task:** Use cross-validation to get a more robust estimate of model performance.

**Business Context:** Before deploying a model, you want to know that it performs consistently across different samples of the data.

**Instructions:**
1. Use 5-fold cross-validation on the banking dataset (shuffle the folds)
2. Calculate the mean and the standard deviation of the R² scores across folds
3. Compare with a simple baseline model that predicts the mean (`DummyRegressor`)
4. Discuss the stability of your model

**Deliverable:** mean and standard deviation of R² across the five folds for the model and for the baseline, and one sentence on stability.


In [ ]:
# Exercise 4: Your code here
# Hint: Use cross_val_score from sklearn.model_selection with cv=KFold(5, shuffle=True, random_state=42)

from sklearn.model_selection import cross_val_score, KFold
from sklearn.dummy import DummyRegressor

# Your solution:


### Exercise 5 (core): Synthetic Data Generation

**Task:** Create your own synthetic dataset with known coefficients and see how well the model recovers them.

**Instructions:**
1. Generate a dataset with 500 observations and 3 features
2. Create a known linear relationship: y = 2*x1 + 3*x2 - 1.5*x3 + noise
3. Add some outliers to make it more realistic
4. Train a linear model and see how well it recovers the true coefficients
5. Experiment with different noise levels. Before you run it, predict: what happens to the coefficient estimates and to R² as the noise grows?

**Deliverable:** a table of true vs estimated coefficients at three noise levels, and one sentence comparing the outcome with your prediction.


In [ ]:
# Exercise 5: Your code here
# Hint: Use np.random functions to generate features and noise
# True coefficients should be [2, 3, -1.5]

np.random.seed(42)  # For reproducibility

# Your solution:


### Exercise 6 (core): Time Series Forecasting Analysis

**Task:** Analyse the Bitcoin forecasting model more deeply.

**Instructions:**
1. Calculate the directional accuracy (how often the model predicts the correct sign of the return) on the test set, and compare it with the share of days on which the return was positive (the accuracy of "always predict up")
2. Create a cumulative returns plot comparing a strategy that follows the model's sign with buy-and-hold
3. Test different lag lengths (1, 3, 5, 10). Choose the lag length on a validation slice of the *training* period (for example its last 25%), not on the test set, and only then report the test result of the chosen lag
4. Discuss the practical implications for trading, including transaction costs

**Deliverable:** the directional accuracy of the chosen model on the test set next to the share of up-days, and one sentence on whether the model has usable skill.


In [ ]:
# Exercise 6: Your code here
# Hint: Directional accuracy = percentage of times sign(predicted) == sign(actual)
# Cumulative returns = cumsum of returns over time

# Your solution:


### Exercise 7 (optional): Model Comparison

**Task:** Compare linear regression with Ridge and Lasso regression on the banking data.

**Instructions:**
1. Implement a Ridge regression model on the banking dataset
2. Implement a Lasso regression model
3. Choose the regularisation strength `alpha` by cross-validation on the training data (`GridSearchCV`), never on the test set
4. Compare test R² and RMSE of the three models and discuss when you would prefer each

**Deliverable:** a table of test R² and RMSE for the three models with the chosen alphas, and one sentence on which you would use here and why.


In [ ]:
# Exercise 7: Your code here
# Hint: Use Ridge and Lasso from sklearn.linear_model; standardise the features first
# Tune alpha with GridSearchCV on the training data

from sklearn.linear_model import Ridge, Lasso

# Your solution:


### Exercise 8 (core): Business Impact Analysis

**Task:** Quantify the business value of the duration prediction model.

**Business Context:** Longer calls might indicate higher customer engagement and conversion probability. The target is log1p(duration) with `duration` in **seconds**, so the thresholds must be on that scale.

**Scenario:**
- Calls shorter than 2 minutes (log1p(duration) < log1p(120) ≈ 4.80): Low engagement
- Calls of 2 to 5 minutes (log1p(duration) between 4.80 and log1p(300) ≈ 5.71): Medium engagement
- Calls longer than 5 minutes (log1p(duration) > 5.71): High engagement
- You want to prioritise follow-up with predicted high-engagement calls

**Instructions:**
1. Classify actual and predicted test-set durations into the three engagement categories
2. Report the share of *actual* calls and the share of *predicted* calls in each category, and the confusion matrix
3. Calculate the accuracy for each engagement level
4. Estimate the business value of a "follow up only with predicted High" strategy against "follow up with everyone", under stated assumptions for cost per call and value per correctly identified high-engagement call

**Deliverable:** the share of test predictions in each band and one sentence on whether this model can be used to rank calls by engagement.


In [ ]:
# Exercise 8: Your code here
# Hint: Use np.where or pd.cut to create engagement categories with thresholds np.log1p(120) and np.log1p(300)
# Calculate the confusion matrix for the three categories

# Your solution:


### Exercise 9 (optional): Advanced Diagnostics

**Task:** Perform advanced model diagnostics to identify potential issues.

**Instructions:**
1. Calculate Cook's distance for the banking model to identify influential observations. Never build the full hat matrix (33,000 x 33,000 would need 8 GB of memory): take the leverages `h` from `OLSInfluence(model_sm).hat_matrix_diag` and use the closed form `D = e**2 / (p * s2) * h / (1 - h)**2` with `e = model_sm.resid`, `p` the number of columns including the constant and `s2 = model_sm.mse_resid`. (`OLSInfluence(...).cooks_distance` computes the same thing but is far too slow at this sample size; if you want to use it, do so on a 5,000-row subsample.)
2. Check for multicollinearity using the variance inflation factor (VIF)
3. Perform the Durbin-Watson test for autocorrelation on the residuals of the **Bitcoin** model, and explain why the statistic is meaningless on the banking model
4. Suggest improvements based on your findings

**Deliverable:** the number of influential observations, the regressors with VIF > 10, the Durbin-Watson statistic of the Bitcoin model, and one sentence on what each implies for the *inference* (not the predictions).


In [ ]:
# Exercise 9: Your code here
# Hint: from statsmodels.stats.outliers_influence import OLSInfluence, variance_inflation_factor
#       from statsmodels.stats.stattools import durbin_watson
#       h = OLSInfluence(model_sm).hat_matrix_diag; cooks_d = model_sm.resid**2 / (p * model_sm.mse_resid) * h / (1 - h)**2

# Your solution:


### Reflection Questions

After completing the exercises, consider these questions:

1. **When might linear regression not be appropriate?**
   - Think about non-linear relationships, categorical outcomes, etc.

2. **How do you balance model complexity with interpretability?**
   - Consider the trade-off between adding features and keeping the model simple

3. **What are the key assumptions of linear regression and why do they matter?**
   - Relate each assumption to potential business consequences if violated

4. **How would you explain R² to a non-technical business stakeholder?**
   - Focus on practical interpretation rather than the formula; use the banking model's R² of about 0.01 and the admission model's R² of about 0.64 as your two examples

5. **In what business scenarios would you prefer RMSE over R² as an evaluation metric?**
   - Think about when absolute prediction errors matter more than explained variance

6. **How might you improve the Bitcoin forecasting model, and what is the most likely outcome of trying?**
   - Consider external factors, different features, or alternative approaches, and what market efficiency implies

### Additional Challenges

For further learning:
- Try implementing linear regression from scratch using matrix operations
- Explore regularization techniques (Ridge, Lasso, Elastic Net) in more detail
- Practice with different datasets (housing prices, stock returns, sales forecasting)
- Learn about advanced time series models (ARIMA, GARCH) for financial data
- Investigate non-linear regression techniques (polynomial, spline regression)

### Key Takeaways

- **One regressor, one sentence.** In this run, GRE score alone explained about 64% of the variation in admission chances, and the slope (about 0.01 per GRE point) is the whole story.
- **Only use what is known at decision time.** The banking model uses pre-call features only; `duration`, `y` and `campaign` are excluded. The same rule made us drop everything observed on day *t* from the Bitcoin regressors.
- **A low R² can be the right answer.** The banking model reached a test R² of about 0.013 and an RMSE of about 0.91 that is within one percent of the standard deviation of the target: call length is mostly decided during the call. That is a finding about the business, not a coding error.
- **Check assumptions with the right tool.** Residuals against fitted values for homoscedasticity, a Q-Q plot for normality, VIFs for multicollinearity (the three macro indicators had VIFs of 30 to 65 in this run, so their coefficients cannot be read separately), and Durbin-Watson or the ACF for autocorrelation, but only where the rows have a time order.
- **Compare with the right benchmark.** Price on lagged price gave a test R² of 0.99 that "yesterday's price" matches without any model. On returns, the model's RMSE was 1.003 times the zero forecast and the test R² was -0.006: no forecasting skill, which is what an efficient market looks like. A lag-1 benchmark would have made the same model look 30% better.
- **Significant is not the same as useful.** Two Bitcoin lags had p-values around 0.01 to 0.02 inside an R² of 0.006. Standard errors and p-values say how precisely a coefficient is estimated, not whether it matters.
